In [1]:
year          = 2023
loc_json = "json/location_params.json"
geo_json      = 'json/geo_params.json'
default_json  = 'json/default_params.json'
config_json   = 'json/UQ_null.json'
epw_path      = '../src/_base/epw/ED-TMYx.2023.epw'
occupancyProfile_csv = 'ETHlib/auxiliary/schedules_el_OFFICE.csv'

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import json

ROOT = Path.cwd().parent.resolve()
RC_SIMULATOR = "ETHlib"
if str(RC_SIMULATOR) not in sys.path:
    sys.path.insert(0, str(RC_SIMULATOR))

from _BR_ import *
from _zone_ import DebugZone as Zone
import supply_system
import emission_system
from radiation import Location, Window
from datetime import datetime

CITYLOC       = json.loads(Path(loc_json).read_text())
latitude_deg = CITYLOC['latitude_deg']
longitude_deg  = CITYLOC['longitude_deg']
EDI            = Location(epwfile_path=epw_path)
GEOMETRY       = json.loads(Path(geo_json).read_text())
DEFAULT_PARAMS = json.loads(Path(default_json).read_text())
occupancyProfile = pd.read_csv(occupancyProfile_csv)

In [3]:
import json
from numbers import Number
def get_numbered_filename(folder, stem, suffix=".json", date_fmt="%Y%m%d"):

    folder = Path(folder)
    date_str = datetime.now().strftime(date_fmt)

    i = 1
    while True:
        filename = f"{stem}_{date_str}_{i:03d}{suffix}"
        path = folder / filename

        if not path.exists():
            return filename

        i += 1

def make_uq_json_from_defaults(
    default_params,
    exclude_params=None,
    fixed_ranges=None,
    N=10,
    seed=42,
    variation=0.10,
    distribution="uniform",
    save_dir=Path(r"json\UQ"),
    output_stem="uq_default_pm10",
    round_digits=6,
):

    if exclude_params is None:
        exclude_params = []

    if fixed_ranges is None:
        fixed_ranges = {}

    exclude_params = set(exclude_params)

    parameters = {}

    for name, value in default_params.items():

        if name in exclude_params:
            continue

        if not isinstance(value, Number):
            continue

        if name in fixed_ranges:
            lower, upper = fixed_ranges[name]
        else:
            lower = value * (1.0 - variation)
            upper = value * (1.0 + variation)

        parameters[name] = {
            "distribution": distribution,
            "lower": round(lower, round_digits),
            "upper": round(upper, round_digits),
        }

    config = {
        "N": N,
        "seed": seed,
        "parameters": parameters,
    }

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = get_numbered_filename(
        folder=save_dir,
        stem=output_stem,
        suffix=".json",
    )

    output_path = save_dir / filename

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print(f"Saved UQ JSON to: {output_path}")

    return config, output_path

exclude_params = []

customised_ranges = {
    "thermal_capacitance_per_floor_area": (80000, 370000)
}

uq_config, uq_path = make_uq_json_from_defaults(
    default_params=DEFAULT_PARAMS,
    exclude_params=exclude_params,
    fixed_ranges=customised_ranges,
    N=10,
    seed=42,
    variation=0.2,
    output_stem="uq_",
)

Saved UQ JSON to: json\UQ\uq__20260610_001.json


In [4]:
DATE = "JUN9"
record_path = Path("json/_run") / f"{DATE}.json"

record = {
    "latest_uq_json": str(uq_path)
}

with open(record_path, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

print(f"Saved latest UQ path record to: {record_path}")

Saved latest UQ path record to: json\_run\JUN9.json
